# Chapter 41: FastAPI Capstone — Build & Deploy — Colab Notebook

This notebook actually runs the FastAPI code that the lesson page (`ch41-fastapi-capstone-deployment.html`) shows as reference-only. FastAPI needs a real ASGI runtime, which Pyodide (the in-browser Python powering the rest of the course) can't provide.

Everything below uses `TestClient` (in-process, no real socket) **except one cell near the end** that starts a genuine `uvicorn` server in a background thread and calls it with a real HTTP request over `127.0.0.1` — the one moment in the whole module where this stops being a simulation.

The app is defined as one self-contained file's worth of code across a few cells (a real project would split it into `main.py`, `database.py`, `auth.py`, `models.py`, `routers/`). It uses in-memory SQLite, so nothing touches your disk.

Run the cells top to bottom. Install dependencies first if needed:

```
!pip install -q fastapi uvicorn httpx sqlalchemy aiosqlite pyjwt bcrypt python-multipart
```

## 41.1 — Assembling the App With APIRouter

In [1]:
from fastapi import APIRouter, FastAPI
from fastapi.testclient import TestClient

demo_router = APIRouter()

@demo_router.get("/health")
def health():
    return {"status": "ok"}

demo_app = FastAPI(title="Router demo")
demo_app.include_router(demo_router, prefix="/candidates", tags=["candidates"])

demo_client = TestClient(demo_app)
print("route paths:", sorted(demo_app.openapi()["paths"]))
r = demo_client.get("/candidates/health")
print(r.status_code, r.json())
print("without the prefix:", demo_client.get("/health").status_code)

route paths: ['/candidates/health']
200 {'status': 'ok'}
without the prefix: 404


## 41.2 — Candidate Scoring API v2 — Putting It All Together

Three cells build the full app. In a real project each would be its own file (`database.py` + `models.py`, `auth.py`, `routers/` + `main.py`) — here they share one namespace so the notebook stays self-contained.

**Cell 1 — database layer and schemas** (Ch 36 Pydantic, Ch 38 SQLAlchemy):

In [2]:
from contextlib import asynccontextmanager
from datetime import datetime, timedelta, timezone
import threading
import time

import bcrypt
from fastapi import (APIRouter, BackgroundTasks, Depends, FastAPI, HTTPException,
                     Request, status)
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
import jwt
from pydantic import BaseModel, ConfigDict, Field
from sqlalchemy import select
from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column
from sqlalchemy.pool import StaticPool


class Base(DeclarativeBase):
    pass


class CandidateRow(Base):
    __tablename__ = "candidates"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str]
    city: Mapped[str]
    score: Mapped[float]


class CandidateIn(BaseModel):
    name: str = Field(min_length=1)
    city: str
    score: float = Field(ge=0, le=1)


class CandidateOut(CandidateIn):
    model_config = ConfigDict(from_attributes=True)
    id: int


async def get_db(request: Request):
    async with request.app.state.sessionmaker() as session:
        yield session


print("database layer + schemas defined")

database layer + schemas defined


**Cell 2 — auth layer** (Ch 39 password hashing, JWT, `get_current_user`):

In [3]:
SECRET_KEY = "capstone-demo-secret-key-change-me-in-production"
ALGORITHM = "HS256"

USERS = {"recruiter": bcrypt.hashpw(b"s3cret-pass", bcrypt.gensalt()).decode()}

oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/auth/token")


def create_token(username: str) -> str:
    claims = {"sub": username, "exp": datetime.now(timezone.utc) + timedelta(minutes=30)}
    return jwt.encode(claims, SECRET_KEY, algorithm=ALGORITHM)


def get_current_user(token: str = Depends(oauth2_scheme)) -> str:
    unauthorized = HTTPException(
        status_code=status.HTTP_401_UNAUTHORIZED,
        detail="Invalid or expired token",
        headers={"WWW-Authenticate": "Bearer"},
    )
    try:
        username = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM]).get("sub")
    except jwt.InvalidTokenError:
        raise unauthorized
    if username not in USERS:
        raise unauthorized
    return username


auth_router = APIRouter()


@auth_router.post("/token")
def login(form: OAuth2PasswordRequestForm = Depends()):
    hashed = USERS.get(form.username)
    if hashed is None or not bcrypt.checkpw(form.password.encode(), hashed.encode()):
        raise HTTPException(status_code=400, detail="Incorrect username or password")
    return {"access_token": create_token(form.username), "token_type": "bearer"}


print("auth layer defined")

auth layer defined


**Cell 3 — routers, middleware, background task, and the app factory** (Ch 36-37 routes and errors, Ch 40 middleware and background tasks):

In [4]:
SENT_EMAILS: list[str] = []


def send_welcome_email(name: str) -> None:
    SENT_EMAILS.append(name)


candidates_router = APIRouter()


@candidates_router.get("", response_model=list[CandidateOut])
async def list_candidates(min_score: float = 0.0, db=Depends(get_db)):
    rows = await db.execute(select(CandidateRow).where(CandidateRow.score >= min_score).order_by(CandidateRow.id))
    return rows.scalars().all()


@candidates_router.get("/{candidate_id}", response_model=CandidateOut)
async def get_candidate(candidate_id: int, db=Depends(get_db)):
    row = await db.get(CandidateRow, candidate_id)
    if row is None:
        raise HTTPException(status_code=404, detail=f"Candidate {candidate_id} not found")
    return row


@candidates_router.post("", response_model=CandidateOut, status_code=201)
async def create_candidate(
    candidate: CandidateIn,
    background_tasks: BackgroundTasks,
    db=Depends(get_db),
    current_user: str = Depends(get_current_user),
):
    row = CandidateRow(**candidate.model_dump())
    db.add(row)
    await db.commit()
    await db.refresh(row)
    background_tasks.add_task(send_welcome_email, row.name)
    return row


@candidates_router.delete("/{candidate_id}", status_code=204)
async def delete_candidate(candidate_id: int, db=Depends(get_db), current_user: str = Depends(get_current_user)):
    row = await db.get(CandidateRow, candidate_id)
    if row is None:
        raise HTTPException(status_code=404, detail=f"Candidate {candidate_id} not found")
    await db.delete(row)
    await db.commit()


SEED = [
    ("Alice", "Hubli", 0.92),
    ("Bob", "Pune", 0.75),
    ("Carol", "Mumbai", 0.81),
    ("Dave", "Delhi", 0.60),
    ("Eve", "Pune", 0.55),
]


async def add_process_time_header(request: Request, call_next):
    start = time.perf_counter()
    response = await call_next(request)
    response.headers["X-Process-Time"] = f"{time.perf_counter() - start:.6f}"
    return response


def create_app() -> FastAPI:
    engine = create_async_engine(
        "sqlite+aiosqlite:///:memory:",
        poolclass=StaticPool,
        connect_args={"check_same_thread": False},
    )
    sessionmaker = async_sessionmaker(engine, expire_on_commit=False)

    @asynccontextmanager
    async def lifespan(app: FastAPI):
        async with engine.begin() as conn:
            await conn.run_sync(Base.metadata.create_all)
        async with sessionmaker() as session:
            session.add_all([CandidateRow(name=n, city=c, score=s) for n, c, s in SEED])
            await session.commit()
        app.state.sessionmaker = sessionmaker
        yield
        await engine.dispose()

    app = FastAPI(title="Candidate Scoring API v2", lifespan=lifespan)
    app.middleware("http")(add_process_time_header)
    app.include_router(auth_router, prefix="/auth", tags=["auth"])
    app.include_router(candidates_router, prefix="/candidates", tags=["candidates"])

    @app.get("/health")
    def health():
        return {"status": "ok"}

    return app


with TestClient(create_app()) as client:
    r = client.get("/candidates", params={"min_score": 0.8})
    print("GET /candidates?min_score=0.8 ->", r.status_code, [c["name"] for c in r.json()])
    print("X-Process-Time header present:", "x-process-time" in r.headers)

GET /candidates?min_score=0.8 -> 200 ['Alice', 'Carol']
X-Process-Time header present: True


## 41.3 — A Full Test Suite

The lesson shows the `pytest` shape (fixtures + one test function per behaviour). This notebook runs the equivalent with plain `assert` statements: each test gets a **fresh app and a fresh in-memory database** from `create_app()`, which is exactly what a `client` fixture does in a real suite.

In [5]:
def login_headers(client, username="recruiter", password="s3cret-pass"):
    r = client.post("/auth/token", data={"username": username, "password": password})
    assert r.status_code == 200, r.text
    return {"Authorization": f"Bearer {r.json()['access_token']}"}


def test_list_filters_by_min_score(client):
    names = [c["name"] for c in client.get("/candidates", params={"min_score": 0.8}).json()]
    assert names == ["Alice", "Carol"]


def test_missing_candidate_is_404(client):
    r = client.get("/candidates/999")
    assert r.status_code == 404
    assert r.json()["detail"] == "Candidate 999 not found"


def test_create_requires_auth(client):
    r = client.post("/candidates", json={"name": "Grace", "city": "Pune", "score": 0.9})
    assert r.status_code == 401


def test_create_validates_body(client):
    r = client.post("/candidates", json={"name": "Grace", "city": "Pune", "score": 1.7}, headers=login_headers(client))
    assert r.status_code == 422


def test_create_succeeds_and_sends_email(client):
    before = len(SENT_EMAILS)
    r = client.post("/candidates", json={"name": "Grace", "city": "Pune", "score": 0.9}, headers=login_headers(client))
    assert r.status_code == 201
    assert r.json()["id"] == 6
    assert SENT_EMAILS[before:] == ["Grace"]


def test_bad_password_is_rejected(client):
    r = client.post("/auth/token", data={"username": "recruiter", "password": "wrong"})
    assert r.status_code == 400


def test_delete_flow(client):
    headers = login_headers(client)
    assert client.delete("/candidates/5", headers=headers).status_code == 204
    assert client.get("/candidates/5").status_code == 404


tests = [v for k, v in sorted(globals().items()) if k.startswith("test_") and callable(v)]
for test in tests:
    with TestClient(create_app()) as fresh_client:
        test(fresh_client)
    print("PASS", test.__name__)
print(f"{len(tests)}/{len(tests)} tests passed")

PASS test_bad_password_is_rejected
PASS test_create_requires_auth


PASS test_create_succeeds_and_sends_email


PASS test_create_validates_body


PASS test_delete_flow
PASS test_list_filters_by_min_score
PASS test_missing_candidate_is_404
7/7 tests passed


## 41.4 — Dockerizing the App

You containerized a trained model in Chapter 31.3 — same idea, bigger app. This cell only **prints** the files a real project would contain (it deliberately doesn't write a `Dockerfile` into your working directory):

In [6]:
requirements = """fastapi
uvicorn
sqlalchemy
aiosqlite
pyjwt
bcrypt
python-multipart
"""

dockerfile = """FROM python:3.12-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt

COPY . .
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
"""

print("# requirements.txt")
print(requirements)
print("# Dockerfile")
print(dockerfile)
print("# build and run:")
print("#   docker build -t candidate-api-v2 .")
print("#   docker run -p 8000:8000 candidate-api-v2")

# requirements.txt
fastapi
uvicorn
sqlalchemy
aiosqlite
pyjwt
bcrypt
python-multipart

# Dockerfile
FROM python:3.12-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt

COPY . .
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

# build and run:
#   docker build -t candidate-api-v2 .
#   docker run -p 8000:8000 candidate-api-v2


## 41.5 — Where To Deploy It

Reference only — no code. Once you have a Dockerfile, **Render / Railway** (point at your GitHub repo, get a public URL), **Fly.io** (global container placement, more configuration) and a **bare VM** (install Docker yourself, put nginx in front) all run the exact same container. Where you run it is a later, separate decision.

## A real server, for once

Everything above used `TestClient`, which calls the app in-process. The cell below starts the same app as a **genuine `uvicorn` server** in a background thread, binds a real socket on `127.0.0.1:8001`, and calls it with `httpx` over real HTTP — the same thing `docker run` would do inside a container.

In [7]:
import httpx
import uvicorn

real_app = create_app()
server = uvicorn.Server(uvicorn.Config(real_app, host="127.0.0.1", port=8001, log_level="warning"))
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()

for _ in range(100):
    if server.started:
        break
    time.sleep(0.1)
else:
    raise RuntimeError("uvicorn did not start within 10 seconds")

try:
    base = "http://127.0.0.1:8001"
    r = httpx.get(f"{base}/health")
    print("GET /health ->", r.status_code, r.json())

    r = httpx.get(f"{base}/candidates", params={"min_score": 0.8})
    print("GET /candidates?min_score=0.8 ->", r.status_code, [c["name"] for c in r.json()])

    token = httpx.post(f"{base}/auth/token", data={"username": "recruiter", "password": "s3cret-pass"}).json()["access_token"]
    r = httpx.post(f"{base}/candidates", json={"name": "Heidi", "city": "Hubli", "score": 0.88},
                   headers={"Authorization": f"Bearer {token}"})
    print("POST /candidates (real HTTP, real JWT) ->", r.status_code, r.json())
finally:
    server.should_exit = True
    server_thread.join(timeout=10)

print("server stopped:", not server_thread.is_alive())

GET /health -> 200 {'status': 'ok'}
GET /candidates?min_score=0.8 -> 200 ['Alice', 'Carol']


POST /candidates (real HTTP, real JWT) -> 201 {'name': 'Heidi', 'city': 'Hubli', 'score': 0.88, 'id': 6}
server stopped: True


## Mini Project — Candidate Scoring API v2, end to end

In [8]:
with TestClient(create_app()) as client:
    checks = {}

    checks["routers assembled (health + candidates + auth)"] = (
        client.get("/health").status_code == 200
        and client.get("/candidates").status_code == 200
        and client.post("/auth/token", data={"username": "recruiter", "password": "s3cret-pass"}).status_code == 200
    )

    checks["writes are auth-protected"] = (
        client.post("/candidates", json={"name": "Ivan", "city": "Delhi", "score": 0.7}).status_code == 401
    )

    headers = login_headers(client)
    created = client.post("/candidates", json={"name": "Ivan", "city": "Delhi", "score": 0.7}, headers=headers)
    checks["authenticated create persists to the database (201)"] = (
        created.status_code == 201 and client.get(f"/candidates/{created.json()['id']}").json()["name"] == "Ivan"
    )

    checks["pydantic validation rejects a bad score (422)"] = (
        client.post("/candidates", json={"name": "Judy", "city": "Pune", "score": 5}, headers=headers).status_code == 422
    )

    checks["background task ran on create"] = "Ivan" in SENT_EMAILS
    checks["timing middleware adds X-Process-Time"] = "x-process-time" in client.get("/health").headers

    for label, ok in checks.items():
        print("PASS" if ok else "FAIL", "-", label)
    print("Project checklist:", "PASSED" if all(checks.values()) else "FAILED")

PASS - routers assembled (health + candidates + auth)
PASS - writes are auth-protected
PASS - authenticated create persists to the database (201)
PASS - pydantic validation rejects a bad score (422)
PASS - background task ran on create
PASS - timing middleware adds X-Process-Time
Project checklist: PASSED


## Course complete

You've built, tested, and (on paper) containerized a real, authenticated, database-backed API — routers, async SQLAlchemy, JWT auth, validation, middleware, background tasks, a test suite, and a real server answering real HTTP. Next stop: the **Interview Prep** tab on the chapter page, then put this project on your GitHub.